<a href="https://colab.research.google.com/github/UjwalaSimma06/Data-Immersion-Wrangling/blob/main/Task_1(Apex).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IMPORT LIBRARIES

In [13]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

STEP 1: LOAD THE DATASET

In [14]:
df = pd.read_csv("/content/raw_sales_data.csv")
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns")
print(f"\n Columns: {list(df.columns)}")

Loaded 20 rows and 14 columns

 Columns: ['order_id', 'customer_name', 'email', 'date_of_birth', 'order_date', 'product', 'category', 'quantity', 'unit_price', 'total_amount', 'city', 'country', 'gender', 'customer_since']


STEP 2: INITIAL DATA PROFILING

In [15]:
print("\n" + "=" * 60)
print("STEP 2: Initial Data Profiling")
print("=" * 60)

print("\n Data Types:")
print(df.dtypes.to_string())

print("\n Missing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0].to_string())

print("\n Duplicate Rows:")
dupes = df.duplicated().sum()
print(f"Found {dupes} duplicate row(s)")

print("\n Basic Statistics (numeric columns):")
print(df.describe().to_string())

print("\n Unique values in categorical columns:")
for col in ['category', 'gender', 'country']:
    print(f"{col}: {df[col].unique()}")


STEP 2: Initial Data Profiling

 Data Types:
order_id           int64
customer_name     object
email             object
date_of_birth     object
order_date        object
product           object
category          object
quantity           int64
unit_price         int64
total_amount       int64
city              object
country           object
gender            object
customer_since    object

 Missing Values:
               Missing Count  Missing %
customer_name              1        5.0
email                      2       10.0

 Duplicate Rows:
Found 0 duplicate row(s)

 Basic Statistics (numeric columns):
         order_id   quantity  unit_price  total_amount
count    20.00000  20.000000     20.0000  2.000000e+01
mean   1010.50000   1.300000  35700.0000  5.404999e+05
std       5.91608   0.571241  22739.0228  2.226767e+06
min    1001.00000   1.000000   2000.0000  4.000000e+03
25%    1005.75000   1.000000  22500.0000  2.875000e+04
50%    1010.50000   1.000000  33500.0000  3.650000e+04


STEP 3: HANDLE DUPLICATES

In [16]:
print("\n" + "=" * 60)
print("STEP 3: Removing Duplicates")
print("=" * 60)
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate row(s). Rows remaining: {after}")


STEP 3: Removing Duplicates
Removed 0 duplicate row(s). Rows remaining: 20


STEP 4: HANDLE MISSING VALUES

In [17]:
print("\n" + "=" * 60)
print("STEP 4: Handling Missing Values")
print("=" * 60)

# email: fill with 'unknown@unknown.com'
df['email'] = df['email'].fillna('unknown@unknown.com')
print("email: filled missing with 'unknown@unknown.com'")

# customer_name: fill with 'Unknown Customer'
df['customer_name'] = df['customer_name'].fillna('Unknown Customer')
print("customer_name: filled missing with 'Unknown Customer'")

print(f"\n Missing values after fix: {df.isnull().sum().sum()}")


STEP 4: Handling Missing Values
email: filled missing with 'unknown@unknown.com'
customer_name: filled missing with 'Unknown Customer'

 Missing values after fix: 0


STEP 5: STANDARDIZE DATE FORMATS

In [18]:
print("\n" + "=" * 60)
print("STEP 5: Standardizing Date Formats")
print("=" * 60)

def parse_date(date_str):
    """Try multiple date formats and return standardized date."""
    formats = ['%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y', '%Y/%m/%d', '%m/%d/%Y']
    for fmt in formats:
        try:
            return datetime.strptime(str(date_str).strip(), fmt)
        except:
            continue
    return pd.NaT

df['order_date'] = df['order_date'].apply(parse_date)
df['date_of_birth'] = df['date_of_birth'].apply(parse_date)
df['customer_since'] = df['customer_since'].apply(parse_date)

print("order_date standardized to YYYY-MM-DD")
print("date_of_birth standardized to YYYY-MM-DD")
print("customer_since standardized to YYYY-MM-DD")


STEP 5: Standardizing Date Formats
order_date standardized to YYYY-MM-DD
date_of_birth standardized to YYYY-MM-DD
customer_since standardized to YYYY-MM-DD


STEP 6: STANDARDIZE CATEGORICAL FIELDS

In [19]:
print("\n" + "=" * 60)
print("STEP 6: Standardizing Categorical Fields")
print("=" * 60)

# Standardize category
df['category'] = df['category'].str.strip().str.title()
print(f"category standardized: {df['category'].unique()}")

# Standardize gender: M/F → Male/Female
gender_map = {'M': 'Male', 'F': 'Female', 'male': 'Male', 'female': 'Female'}
df['gender'] = df['gender'].str.strip()
df['gender'] = df['gender'].replace(gender_map)
df['gender'] = df['gender'].str.title()
print(f"gender standardized: {df['gender'].unique()}")

# Standardize city names
df['city'] = df['city'].str.strip().str.title()
df['country'] = df['country'].str.strip().str.title()
print(f"city and country formatted consistently")


STEP 6: Standardizing Categorical Fields
category standardized: ['Electronics' 'Appliances']
gender standardized: ['Male' 'Female']
city and country formatted consistently


STEP 7: HANDLE OUTLIERS

In [20]:
print("\n" + "=" * 60)
print("STEP 7: Handling Outliers")
print("=" * 60)

# Check total_amount outliers using IQR
Q1 = df['total_amount'].quantile(0.25)
Q3 = df['total_amount'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['total_amount'] < lower) | (df['total_amount'] > upper)]
print(f"   Outliers found in total_amount: {len(outliers)} row(s)")
print(f"   IQR bounds: [{lower:.0f} – {upper:.0f}]")

# Flag outliers instead of removing them
df['is_outlier'] = ((df['total_amount'] < lower) | (df['total_amount'] > upper))
print("Outliers flagged in 'is_outlier' column (not deleted)")


STEP 7: Handling Outliers
   Outliers found in total_amount: 2 row(s)
   IQR bounds: [-10625 – 94375]
Outliers flagged in 'is_outlier' column (not deleted)


STEP 8: FEATURE ENGINEERING

In [21]:
print("\n" + "=" * 60)
print("STEP 8: Feature Engineering (Creating New Columns)")
print("=" * 60)

today = datetime.today()

# Customer Age from date_of_birth
df['customer_age'] = df['date_of_birth'].apply(
    lambda dob: int((today - dob).days / 365.25) if pd.notnull(dob) else np.nan
)
print("customer_age: derived from date_of_birth")

# Age Group / Customer Segment
def age_segment(age):
    if pd.isna(age): return 'Unknown'
    if age < 18: return 'Minor'
    elif age < 25: return 'Gen Z'
    elif age < 40: return 'Millennial'
    elif age < 55: return 'Gen X'
    else: return 'Baby Boomer'

df['age_segment'] = df['customer_age'].apply(age_segment)
print("age_segment: Minor / Gen Z / Millennial / Gen X / Baby Boomer")

# Customer Tenure (years since customer_since)
df['tenure_years'] = df['customer_since'].apply(
    lambda d: round((today - d).days / 365.25, 1) if pd.notnull(d) else np.nan
)
print("tenure_years: years since customer_since date")

# Revenue per unit
df['revenue_per_unit'] = (df['total_amount'] / df['quantity']).round(2)
print("revenue_per_unit: total_amount / quantity")

# Order month and year
df['order_month'] = df['order_date'].dt.month
df['order_year'] = df['order_date'].dt.year
print("order_month and order_year: extracted from order_date")

# High-value customer flag (total_amount > 50000)
df['is_high_value'] = df['total_amount'] > 50000
print("is_high_value: True if total_amount > ₹50,000")

# Minor flag (for compliance)
df['is_minor'] = df['customer_age'] < 18
print("is_minor: True if customer_age < 18 (compliance flag)")


STEP 8: Feature Engineering (Creating New Columns)
customer_age: derived from date_of_birth
age_segment: Minor / Gen Z / Millennial / Gen X / Baby Boomer
tenure_years: years since customer_since date
revenue_per_unit: total_amount / quantity
order_month and order_year: extracted from order_date
is_high_value: True if total_amount > ₹50,000
is_minor: True if customer_age < 18 (compliance flag)


STEP 9: FINAL VALIDATION

In [22]:
print("\n" + "=" * 60)
print("STEP 9: Final Validation")
print("=" * 60)
print(f"Total Rows: {len(df)}")
print(f"Total Columns: {len(df.columns)}")
print(f"Missing Values Remaining: {df.isnull().sum().sum()}")
print(f"Columns in final dataset: {list(df.columns)}")

print("\n Sample of final cleaned data:")
print(df[['order_id', 'customer_name', 'customer_age', 'age_segment',
          'category', 'gender', 'total_amount', 'is_outlier',
          'tenure_years', 'is_high_value']].head(10).to_string())


STEP 9: Final Validation
Total Rows: 20
Total Columns: 23
Missing Values Remaining: 0
Columns in final dataset: ['order_id', 'customer_name', 'email', 'date_of_birth', 'order_date', 'product', 'category', 'quantity', 'unit_price', 'total_amount', 'city', 'country', 'gender', 'customer_since', 'is_outlier', 'customer_age', 'age_segment', 'tenure_years', 'revenue_per_unit', 'order_month', 'order_year', 'is_high_value', 'is_minor']

 Sample of final cleaned data:
   order_id customer_name  customer_age age_segment     category  gender  total_amount  is_outlier  tenure_years  is_high_value
0      1001    John Smith            36  Millennial  Electronics    Male         75000       False           6.2           True
1      1002  Priya Sharma            30  Millennial  Electronics  Female         50000       False           6.9          False
2      1003   Rahul Verma            37  Millennial  Electronics    Male          6000       False           5.3          False
3      1004  Anjali Pa

STEP 10: EXPORT CLEANED DATASET

In [23]:
print("\n" + "=" * 60)
print("STEP 10: Exporting Cleaned Dataset")
print("=" * 60)
df.to_csv("cleaned_sales_data.csv", index=False)
print("Saved: cleaned_sales_data.csv")

print("\n" + "=" * 60)
print("DATA WRANGLING COMPLETE!")
print("=" * 60)


STEP 10: Exporting Cleaned Dataset
Saved: cleaned_sales_data.csv

DATA WRANGLING COMPLETE!
